In [2]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open('../data/hotpot_train_v1.1.json', 'r') as json_file:
        data_json = json.load(json_file)
    





In [3]:
def get_data_as_dataframe(data_json):
    data_preprocessed = []
    for data in data_json:
        context = data['context']
        supporting_factsArray = data['supporting_facts']
        supporting_facts = [str(supporting_factsTupel[0]) for supporting_factsTupel in supporting_factsArray]
        for contexA in context:
                title = str(contexA[0])
                sentences = contexA[1]
                sentence = ""
               
                for sentenceX in sentences:
                    sentence += sentenceX

                data_preprocessed.append(
                    {
                        'question_id': data['_id'],
                        'question': data['question'],
                        'answer': data['answer'],
                        'level': data['level'],
                        'type': data['type'],
                        'sentence': sentence,
                        'is_supporting_sentence': title in supporting_facts
                    }
                )
    return pd.DataFrame(data_preprocessed)
data = get_data_as_dataframe(data_json)


In [4]:
question_groups = data.groupby('question_id')

def get_question_stats(group):
    """Get statistics for a question group"""
    supporting_count = group['is_supporting_sentence'].value_counts().get(True, 0)  # Changed to use value_counts
    unsupporting_count = group['is_supporting_sentence'].value_counts().get(False, 0)  # Changed to use value_counts
    return {
        'total_sentences': len(group),
        'supporting_sentences': supporting_count,
        'unsupporting_sentences': unsupporting_count,
        'level': group['level'].iloc[0],
        'question': group['question'].iloc[0],
        'answer': group['answer'].iloc[0],
        'type': group['type'].iloc[0]
    }

question_stats = []
for qid, group in question_groups:
    stats = get_question_stats(group)
    stats['question_id'] = qid
    if stats['unsupporting_sentences'] != 8:
        #remove qid from data
        data = data[data['question_id'] != qid]
    else: 
        question_stats.append(stats)

question_stats_df = pd.DataFrame(question_stats)

print("\nQuestion stats summary:")
print("Total questions:", len(question_stats_df))
print("\nSupporting sentences distribution:")
print(question_stats_df['supporting_sentences'].value_counts().sort_index())
print("\nUnsupporting sentences distribution:")
print(question_stats_df['unsupporting_sentences'].value_counts().sort_index())



Question stats summary:
Total questions: 89609

Supporting sentences distribution:
supporting_sentences
2    89609
Name: count, dtype: int64

Unsupporting sentences distribution:
unsupporting_sentences
8    89609
Name: count, dtype: int64


In [ ]:

# Anzahl der gewünschten Samples pro Typ und Level
n_samples_per_type_level = 100

# Sampling der Daten
sampled_questions = (
    question_stats_df
    .groupby(['level', 'type'], group_keys=False)  # Gruppiere nach Level und Typ
    .apply(lambda x: x.sample(n=min(len(x), n_samples_per_type_level)))  # Sample bis zu n_samples_per_type_level
    .reset_index(drop=True)
)

data_final = []
for qid in sampled_questions['question_id']:
    contexts = data[data['question_id'] == qid]
    
    contextslist = contexts['sentence'].tolist()
    #get one not supporting sentence
    not_supporting_sentences = contexts[contexts['is_supporting_sentence'] == False]['sentence'].tolist()
    support__sentences = contexts[contexts['is_supporting_sentence'] == True]['sentence'].tolist()
    currentquestion = sampled_questions[sampled_questions['question_id'] == qid]
    data_final.append({"question_id": qid,
                        "multi_hop_question": currentquestion['question'].iloc[0],
                        "multi_hop_answer": currentquestion['answer'].iloc[0],
                        "level": currentquestion['level'].iloc[0],
                        "type": currentquestion['type'].iloc[0],
                        "contextList": contextslist,
                        "supporting__sentences": support__sentences,
                        "not_supporting_sentences": not_supporting_sentences,
                        
                       })
data_final = pd.DataFrame(data_final, index= None)
data_final = data_final.reset_index(drop=True)
#shuffel data_final

#store as json
with open('./static_qas/hotpot_preprocessed_neu.json', 'w') as json_file:
    json.dump(data_final.to_dict(orient='records'), json_file, indent=4)


/tmp/ipykernel_87291/2814246372.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), n_samples_per_type_level)))  # Sample bis zu n_samples_per_type_level


In [15]:
print("\nDataset Statistics:")
print(f"Number of questions: {len(sampled_questions)}")
print("\nDifficulty level distribution:")
print(sampled_questions['level'].value_counts())




level_medium = sampled_questions[sampled_questions['level'] == 'medium']
level_easy = sampled_questions[sampled_questions['level'] == 'easy']
level_hard = sampled_questions[sampled_questions['level'] == 'hard']

types_bridge = sampled_questions[sampled_questions['type'] == 'bridge']
types_comparison = sampled_questions[sampled_questions['type'] == 'comparison'] 
print("\nDataset Statistics:")
print(f"Number of questions: {len(sampled_questions)}")
print("Numeber easy: ", len(level_easy))
print("Numeber medium: ", len(level_medium))
print("Numeber hard: ", len(level_hard))
print("Numeber bridge: ", len(types_bridge))
print("Numeber comparison: ", len(types_comparison))
print(sampled_questions.head(3))






Dataset Statistics:
Number of questions: 600

Difficulty level distribution:
level
easy      200
hard      200
medium    200
Name: count, dtype: int64

Dataset Statistics:
Number of questions: 600
Numeber easy:  200
Numeber medium:  200
Numeber hard:  200
Numeber bridge:  300
Numeber comparison:  300
   total_sentences  supporting_sentences  unsupporting_sentences level  \
0               10                     2                       8  easy   
1               10                     2                       8  easy   
2               10                     2                       8  easy   

                                            question  \
0  how is No. 1458 Flight RAF and Hawker Hurrican...   
1  What other films is the actor known for playin...   
2  What is the most important building in the Nat...   

                                              answer    type  \
0                                           Aircraft  bridge   
1  Warrior" (2011), "The Grey" (2012), "End of 